# DeepSeek-OCR-2 inference — works on T4 and RTX 50-series (Blackwell)

**Why the official snippet breaks on your hardware:**

- `flash_attention_2` requires compute capability >= 8.0 (Ampere+). **T4 (cc 7.5)** does not support it at all — the model load will crash.
- **RTX 50-series (Blackwell, cc 12.0)** is too new for the `flash-attn==2.7.3` wheel pinned in the model card — it wasn't compiled against sm_120, so `pip install flash-attn==2.7.3` either fails to build or installs a kernel that doesn't match your arch.

This notebook detects which of the two GPUs you're on and picks the right attention backend automatically: `sdpa` on T4 (flash-attn is a non-starter there), and on RTX 50 it tries `flash_attention_2` only if a matching wheel actually imports, otherwise falls back to `sdpa` — with `eager` as the final safety net either way. Both GPUs handle `bfloat16` fine, so dtype isn't an issue here.

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

# Core deps pinned to versions DeepSeek-OCR-2 was validated against.
pip_install(
    'torch==2.6.0', 'torchvision',
    'transformers==4.46.3', 'tokenizers==0.20.3',
    'einops', 'addict', 'easydict', 'accelerate', 'Pillow'
)
print('Base deps installed.')

In [ ]:
import torch

assert torch.cuda.is_available(), 'No CUDA GPU visible — check your runtime/accelerator setting.'

device_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
cc = major + minor / 10
print(f'GPU: {device_name}  |  compute capability: {major}.{minor}')

# Both T4 (cc 7.5) and RTX 50-series Blackwell (cc 12.0) handle bf16 fine.
dtype = torch.bfloat16
print(f'Using dtype: {dtype}')

# --- Decide attention backend ---
# T4 (cc 7.5): flash-attn is unsupported on Turing -> never attempt it, go straight to sdpa.
# RTX 50 (Blackwell, cc 12.0): flash-attn==2.7.3 wheel predates sm_120 support, so we try it
#   but only trust it if it actually builds/imports; otherwise sdpa (PyTorch's own fused
#   kernels still work great on Blackwell even without the flash-attn package).
attn_impl = 'sdpa'  # safe default for both GPUs

if cc >= 9.0:  # Blackwell / very new arch -> worth trying flash-attn, but verify it imports
    try:
        pip_install('flash-attn==2.7.3', '--no-build-isolation')
        import flash_attn  # noqa: F401
        attn_impl = 'flash_attention_2'
        print('flash-attn installed and importable -> using flash_attention_2')
    except Exception as e:
        print(f'flash-attn install/import failed on this arch ({e}); falling back to sdpa')
        attn_impl = 'sdpa'
else:  # T4 and similar pre-Ampere/Ampere cards
    print(f'cc {cc} (T4) -> flash-attn not supported, using sdpa')

print(f'Final attention implementation: {attn_impl}')

In [ ]:
from transformers import AutoModel, AutoTokenizer
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
model_name = 'deepseek-ai/DeepSeek-OCR-2'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

def load_model(attn_impl):
    m = AutoModel.from_pretrained(
        model_name,
        _attn_implementation=attn_impl,
        trust_remote_code=True,
        use_safetensors=True,
    )
    return m.eval().cuda().to(dtype)

try:
    model = load_model(attn_impl)
except Exception as e:
    print(f'Load with {attn_impl} failed ({e}); retrying with eager attention')
    attn_impl = 'eager'
    model = load_model(attn_impl)

print(f'Model loaded on {device_name} with attn_implementation={attn_impl}, dtype={dtype}')

## Run inference

Adjust `image_file` to your actual path. `base_size`/`image_size` of 1024/768 is the default "Gundam" crop-mode config from the model card; drop to `base_size=640, image_size=640, crop_mode=False` if you hit OOM on T4/P100 (16GB/12GB cards).

In [ ]:
prompt = "<image>\n<|grounding|>Convert the document to markdown. "
image_file = '/kaggle/input/datasets/ayoubcherguelaine/ocr-base-image-for-test/Screenshot from 2025-11-06 14-38-54.png'
output_path = '/kaggle/working/'

os.makedirs(output_path, exist_ok=True)

with torch.inference_mode():
    res = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=image_file,
        output_path=output_path,
        base_size=1024,
        image_size=768,
        crop_mode=True,
        save_results=True,
    )

print(res)

## If you still OOM on T4 (16GB)

Run this lower-memory config instead of the cell above:
```python
res = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=image_file,
    output_path=output_path,
    base_size=640,
    image_size=640,
    crop_mode=False,   # disables the multi-crop "Gundam" mode, biggest memory saver
    save_results=True,
)
```
RTX 50-series cards have plenty of VRAM (16GB+) so the default `base_size=1024, crop_mode=True` config should run fine without this.